# LawyerGPT — Colab inference

RAG-инференс без Streamlit / Postgres / Docker.

**Перед запуском:**
1. `Runtime` → `Change runtime type` → **GPU (T4)**.
2. В ячейке 2 укажи URL репозитория.
3. В ячейке 4 — креды X5 API.

Первый прогон займёт ~10–25 минут (скачивание BGE-M3 + заполнение Qdrant эмбеддингами всех статей). При повторных запусках Qdrant сохраняется в `./qdrant_local/` — фактически мгновенный старт, если не пересоздавать рантайм.

In [ ]:
# 1) Проверка GPU
!nvidia-smi | head -n 15

In [ ]:
# 2) Клонируем репозиторий
REPO_URL = 'https://github.com/<your-user>/LawyerGPT.git'  # ← подставь свой URL

!git clone $REPO_URL
%cd LawyerGPT

In [ ]:
# 3) Зависимости (torch уже стоит в Colab — не переустанавливаем)
!pip install -q \
    langchain==0.3.25 \
    langchain-openai==0.3.18 \
    langchain-text-splitters==0.3.8 \
    openai==1.82.0 \
    qdrant-client==1.14.2 \
    sentence-transformers==4.1.0 \
    sqlalchemy==2.0.41 \
    pydantic==2.11.5 \
    transformers==4.52.3 \
    tokenizers==0.21.1 \
    tiktoken==0.9.0 \
    python-dotenv \
    PyYAML \
    tqdm \
    hf_transfer

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # быстрее качаем BGE-M3

In [ ]:
# 4) Креды X5 API + config.yaml
import os, yaml

X5_API_BASE_URL = 'https://<x5-host>/v1'   # ← подставь
X5_API_KEY      = '<x5-api-key>'           # ← подставь
X5_MODEL        = '<model-name>'           # ← подставь

os.environ['X5_API_BASE_URL'] = X5_API_BASE_URL
os.environ['X5_API_KEY']      = X5_API_KEY
os.environ['X5_MODEL']        = X5_MODEL

config = {
    'base_url': X5_API_BASE_URL,
    'model':    X5_MODEL,
    'password': X5_API_KEY,
    'rag_settings': {
        'mode': 'standard',
        'answer_temperature': 0.7,
        'router_enable': True,
        'router_temperature': 0.7,
        'db_enable': True,
        'chunk_count': 10,
        'reranker_enable': True,
        'reranker_temperature': 0.5,
        'chain_of_thoughts': False,
        'chat_id': 0,
    },
}
with open('config.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True)

# быстрая проверка доступности X5 endpoint из Colab
import urllib.parse, urllib.request, socket
host = urllib.parse.urlparse(X5_API_BASE_URL).hostname
try:
    socket.gethostbyname(host)
    print(f'DNS OK для {host}')
except Exception as e:
    print(f'DNS FAIL для {host}: {e} — Colab не сможет достучаться до X5')

In [ ]:
# 5) In-memory замены Postgres и истории чата (вместо БД)
from typing import Dict

class InMemoryDB:
    def __init__(self):
        self._codes: Dict[int, dict] = {}
        self._laws:  Dict[int, dict] = {}

    def add_code_atricle(self, article, article_id):
        self._codes[article_id] = {
            'codes_id':      article['codes_id'],
            'chapter_num':   article['chapter_num'],
            'chapter_title': article['chapter_title'],
            'article_num':   article['article_num'],
            'title':         article['title'],
            'text':          article['text'],
            'comments':      article.get('comments', '') or '',
            'additional':    (article.get('additional') or '') + '\n' + (article.get('additional_2') or ''),
            'court_links':   article.get('court_links', []) or [],
        }

    def add_law_atricle(self, article, article_id):
        self._laws[article_id] = {
            'law_id':        article['law_id'],
            'chapter_num':   article.get('chapter_num') or '',
            'chapter_title': article.get('chapter_title') or '',
            'article_num':   article['article_num'],
            'title':         article['title'],
            'text':          article['text'],
            'comments':      article.get('comments', '') or '',
            'additional':    (article.get('additional') or '') + '\n' + (article.get('additional_2') or ''),
            'court_links':   article.get('court_links', []) or [],
        }

    def get_code_article(self, article_id): return self._codes.get(article_id)
    def get_law_article(self,  article_id): return self._laws.get(article_id)


class InMemoryHistory:
    def add_message(self, chat_id, role, text): pass
    def get_messages(self, chat_id, limit=None): return []
    def delete_messages_by_chat_id(self, chat_id): pass


# Патчим до того, как rag_agent импортирует эти модули
import database.psql_base   as _psql
import database.history_base as _hist
_psql.PostgresBase = InMemoryDB
_hist.HistoryBase  = InMemoryHistory
print('PostgresBase / HistoryBase → in-memory ✓')

In [ ]:
# 6) Патч QdrantLegalRAG: embedded Qdrant + CUDA для BGE-M3
import database.vbase as _vb
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

def colab_init(self, host=None, port=None, collection_prefix='legal', timeout=1000.0):
    self.client = QdrantClient(path='./qdrant_local')
    self.encoder = SentenceTransformer(
        'BAAI/bge-m3',
        device='cuda',
        model_kwargs={'low_cpu_mem_usage': False},
    )
    self.constitution_collection = f'{collection_prefix}_constitution'
    self.codes_collection        = f'{collection_prefix}_codes'
    self.laws_collection         = f'{collection_prefix}_laws'
    self.postgres_base = _psql.PostgresBase()
    self.text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600, chunk_overlap=50,
        length_function=len, separators=['\n\n', '\n', '.', ' ', ''],
    )
    self.vector_dimension = 1024
    self._initialize_collections()

_vb.QdrantLegalRAG.__init__ = colab_init
print('QdrantLegalRAG → embedded + CUDA ✓')

In [ ]:
# 7) Создаём агента (первый раз — долго: качается BGE-M3 + заливаются эмбеддинги)
from src.rag_agent import RagAgent

s = RagAgent()
print('RagAgent готов')

In [ ]:
# 8) Инференс
text_question = (
    'Истрин отказался подписывать протокол об административном правонарушении, '
    'не согласившись с его содержанием, и потребовал выдать копию под расписку. '
    'Начальник погранзаставы отказал, сославшись на наличие формулировки '
    '«С протоколом ознакомлен, согласен». Правомерны ли действия начальника? '
    'Какие права имеет лицо, в отношении которого составлен протокол?'
)

print(s(text_question)['answer'])

In [ ]:
# 9) Ещё несколько примеров
for q in [
    'Что будет, если я травмировал в драке человека?',
    'А если я оборонялся?',
    'А если я был в состоянии аффекта?',
]:
    print('Q:', q)
    print('A:', s(q)['answer'])
    print('-' * 80)